### Query Translation - HyDE
Query translation sits at the first stage of an advanced RAG pipeline. The goal of query translation is to take the input user question and to translate it in some way as to improve retrival.
* **Query re-writing** is one approach - we discussed techniques, such as _RAG Fusion_ and _Multi Query_ in previous notebooks. The idea there is to essentially modify the user-query (generate various variations of the query), so we can capture additional perspectives of what the user intends to ask.
* **Query Decomposition** is another approach, where we break down a complex query into various sub-queries (or sub-questions) which are then independently executed against the LLM and results are then combined together.

### What is HyDE?
HYDE is an interesting approach that takes adbantage of a very simple idea. The basic RAG flow takes a question and embeds it; takes a document & embeds it and looks for similarity between an embeded document & the embedded question. However, the question & document are very dis-similar. A document can very large and complex - may come from _dense_ publications (such as PDFs) and other sources, whereas questions are usually short & terse and could be ill-worded from users.

The intuition behind HyDE is take questions and map them into document space using a hypothetical document (or by generating a hypothetical document). The idea is shown visually in the diagram below - in principle, for certain cases, a hypothetics document is _closer_ to desired document you want to retrieve from the high dimension vector space of the embedding than the sparse raw input question. This is a means of translating raw questions into hypotheticsl documents, which are better suited for retrieval.
 
![HYDE](images/hyde.png)


In [19]:
import bs4, os
import pathlib
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown
from IPython.display import display, Markdown

from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# since we are using Gemini, we'll use Google embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

In [20]:
# load API keys from .env files
load_dotenv(override=True)
console = Console()

In [21]:
# create our LLM - we'll be using Gemini-2.5-flash
llm = init_chat_model("google_genai:gemini-2.5-flash", temperature=0.0)
faiss_store = pathlib.Path(os.getcwd()) / "faiss_index_rag_qd"

In [22]:
def create_or_load_embeddings():
    """creates if not available or loads from disk a FAISS embedding"""
    if not faiss_store.exists():
        # in this example we'll load document from a URL
        web_url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
        console.print(
            f"[yellow]Loading document from URL {web_url}. Please wait...[/yellow]"
        )
        loader = WebBaseLoader(
            web_paths=(web_url,),
            bs_kwargs=dict(
                parse_only=bs4.SoupStrainer(
                    class_=("post-content", "post-title", "post-header")
                )
            ),
        )
        blog_docs = loader.load()

        console.print(f"[blue]Loaded {len(blog_docs)} documents from URL[/blue]")
        console.print(
            f"[blue]Metadata of first document: {blog_docs[0].metadata}[/blue]"
        )
        console.print(
            f"[blue]First 200 chars of first document: {blog_docs[0].page_content[:200]}[/blue]"
        )

        # split document into chunks of 1000 chars with 200 chars overlap
        console.print(f"[yellow]Chunking the PDF. Please wait...[/yellow]")

        text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
            chunk_size=300, chunk_overlap=50
        )

        # Make splits
        splits = text_splitter.split_documents(blog_docs)
        console.print(f"[blue]Created {len(splits)} chunks[/blue]")

        # save to embeddings
        console.print("[yellow]Creating embeddings. Please wait...[/yellow]")
        # Use a Gemini embedding model that is suitable for retrieval.
        # It is important to match the model to the task.
        embeddings = GoogleGenerativeAIEmbeddings(
            model="models/text-embedding-004",
            task_type="retrieval_document",
        )
        vector_store = FAISS.from_documents(documents=splits, embedding=embeddings)
        retriever = vector_store.as_retriever()
        vector_store.save_local(str(faiss_store))
        console.print(
            f"[yellow]Local embeddings created at {str(faiss_store)}[/yellow]"
        )
    else:
        console.print(
            f"[yellow]Loading existing embeddings from {str(faiss_store)}[/yellow]"
        )
        embeddings = GoogleGenerativeAIEmbeddings(
            model="models/text-embedding-004",
            task_type="retrieval_document",
        )
        vector_store = FAISS.load_local(
            str(faiss_store), embeddings, allow_dangerous_deserialization=True
        )
        retriever = vector_store.as_retriever()

    return retriever

In [23]:
retriever = create_or_load_embeddings()

Loading existing embeddings from 
c:\Users\BHOBEMRMANISHJAGDISH\Dev\code\git_projects\learning_langchain\src\langchain_tutorial\faiss_index_rag_qd

In [24]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = """Please write a scienticic paper passage to answer the following question: 
Question: {question}
Passage: """

prompt_hyde = ChatPromptTemplate.from_template(prompt_template)

generate_docs_for_retrieval = prompt_hyde | llm | StrOutputParser()

question = "What is task decomposition for LLM Agents?"
generated_doc = generate_docs_for_retrieval.invoke({"question": question})
print(generated_doc)

## Task Decomposition in Large Language Model Agents

Task decomposition, in the context of Large Language Model (LLM) agents, refers to the process of systematically breaking down a complex, high-level objective into a series of smaller, more manageable, and often interdependent sub-tasks. This strategy is fundamental for enabling LLM agents to address intricate problems that would otherwise exceed their inherent capabilities, transforming an intractable problem into a sequence of solvable steps.

The necessity for task decomposition arises from several limitations of current LLMs. Despite their advanced reasoning and generation abilities, LLMs possess finite context windows, are susceptible to 'hallucination' when performing multi-step reasoning over extended sequences, and can struggle with long-term planning or complex logical inference that requires deep, sequential thought. By decomposing a task, the cognitive load on the LLM at each step is significantly reduced, allowing it to 

So we have generated a hypothetical document, which hopefully maps close to relevant documents in the larger vector embedding space. Now we can use this document to do a similarity search

In [25]:
retrieval_chain = generate_docs_for_retrieval | retriever
retrieved_docs = retrieval_chain.invoke({"question": question})
print(retrieved_docs)

[Document(id='5052b9da-f4e5-4951-8f11-2957eda4d037', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Component One: Planning#\nA complicated task usually involves many steps. An agent needs to know what they are and plan ahead.\nTask Decomposition#\nChain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the model’s thinking process.\nTree of Thoughts (Yao et al. 2023) extends CoT by exploring multiple reasoning possibilities at each step. It first decomposes the problem into multiple thought steps and generates multiple thoughts per step, creating a tree structure. The search process can be BFS (breadth-first search) or DFS (depth-f

In [26]:
# build context
context = ""
for doc in retrieved_docs:
    context += doc.page_content + "\n\n"

Now you RAG from the retrieved documents :)

In [27]:
template = """Answer the following question based on the provided context.

{context}

Question: {question}"""

prompt_template = ChatPromptTemplate.from_template(template)

final_chain = prompt_template | llm | StrOutputParser()
final_response = final_chain.invoke({"context": context, "question": question})
print(display(Markdown(final_response)))

For LLM Agents, task decomposition is the process where the agent breaks down large, complicated tasks into smaller, simpler, and more manageable subgoals or steps. This enables efficient handling of complex tasks and enhances the model's performance.

It can be achieved through various methods:
*   **Chain of Thought (CoT)**: Instructing the model to "think step by step" to decompose hard tasks into smaller, simpler steps.
*   **Tree of Thoughts (ToT)**: Extending CoT by exploring multiple reasoning possibilities at each step, decomposing the problem into multiple thought steps and generating multiple thoughts per step.
*   **LLM with simple prompting**: Using prompts like "Steps for XYZ.\n1." or "What are the subgoals for achieving XYZ?".
*   **Task-specific instructions**: For example, "Write a story outline" for writing a novel.
*   **Human inputs**.

None
